<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - `examples.py` simulation

## Path Constants

Define the project name.

In [1]:
PROJECT = "cl_st2_ph2_arianne"

In [2]:
from pathlib import Path

SCORES_FILE = Path(f"sas/output_{PROJECT}/{PROJECT}_scores_only.tsv")
MEANS_PATTERN = f"sas/output_{PROJECT}/means_group_f{{dim}}.tsv"
FILE_IDS_PATH = Path("file_ids.txt")

## Load factor scores

In [3]:
import pandas as pd

scores_df = pd.read_csv(SCORES_FILE, sep="\t")
scores_df = scores_df.rename(columns={"filename": "file_id"})

scores_df

,file_id,source,model,prompt,group,fac1,fac2,fac3,fac4,v000001,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
0,t000001,ai,gemini,persona,persona_gemini,7,6,0,2,0,...,1,0,0,0,0,0,0,0,0,0
1,t000002,ai,gemini,persona,persona_gemini,3,7,5,1,0,...,1,0,0,0,0,0,0,0,0,0
2,t000003,ai,gemini,persona,persona_gemini,10,8,0,0,0,...,1,0,0,0,1,0,0,0,0,0
3,t000004,ai,gemini,persona,persona_gemini,4,7,6,0,0,...,1,0,0,0,0,0,0,0,0,0
4,t000005,ai,gemini,persona,persona_gemini,5,7,5,3,0,...,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,t003996,human,human,human,human,82,21,8,1,0,...,1,1,1,1,1,0,1,0,0,0
3996,t003997,human,human,human,human,46,12,3,1,0,...,1,1,1,0,1,0,0,0,0,1
3997,t003998,human,human,human,human,39,8,4,3,0,...,1,0,0,0,1,0,0,0,0,0
3998,t003999,human,human,human,human,99,22,21,14,0,...,1,0,1,0,1,0,0,0,0,0


## Load file ID mapping

In [4]:
file_ids_df = pd.read_csv(
    FILE_IDS_PATH,
    sep=" ",
    names=["file_id", "group_filename"],
)

file_ids_df.head()

,file_id,group_filename
0,t000001,t001_gemini.txt
1,t000002,t002_gemini.txt
2,t000003,t003_gemini.txt
3,t000004,t004_gemini.txt
4,t000005,t005_gemini.txt


## Merge factor scores with file ID mapping

In [5]:
scores_file_ids_df = scores_df.merge(file_ids_df, on="file_id", how="left")
scores_file_ids_df = scores_file_ids_df[
    ["file_id", "group_filename"]
    + [col for col in scores_file_ids_df.columns if col not in ["file_id", "group_filename"]]
    ]

scores_file_ids_df


,file_id,group_filename,source,model,prompt,group,fac1,fac2,fac3,fac4,...,v000961,v000962,v000963,v000964,v000965,v000966,v000967,v000968,v000969,v000970
0,t000001,t001_gemini.txt,ai,gemini,persona,persona_gemini,7,6,0,2,...,1,0,0,0,0,0,0,0,0,0
1,t000002,t002_gemini.txt,ai,gemini,persona,persona_gemini,3,7,5,1,...,1,0,0,0,0,0,0,0,0,0
2,t000003,t003_gemini.txt,ai,gemini,persona,persona_gemini,10,8,0,0,...,1,0,0,0,1,0,0,0,0,0
3,t000004,t004_gemini.txt,ai,gemini,persona,persona_gemini,4,7,6,0,...,1,0,0,0,0,0,0,0,0,0
4,t000005,t005_gemini.txt,ai,gemini,persona,persona_gemini,5,7,5,3,...,0,1,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,t003996,t995_human.txt,human,human,human,human,82,21,8,1,...,1,1,1,1,1,0,1,0,0,0
3996,t003997,t996_human.txt,human,human,human,human,46,12,3,1,...,1,1,1,0,1,0,0,0,0,1
3997,t003998,t997_human.txt,human,human,human,human,39,8,4,3,...,1,0,0,0,1,0,0,0,0,0
3998,t003999,t998_human.txt,human,human,human,human,99,22,21,14,...,1,0,1,0,1,0,0,0,0,0


## Load group means

In [6]:
import re

factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

means_dfs = {}

for factor_col in factor_cols:
    fac_num = int(factor_col.replace("fac", ""))
    means_file = Path(MEANS_PATTERN.format(dim=fac_num))

    if not means_file.exists():
        raise FileNotFoundError(f"Means file not found: {means_file}")

    means_df = pd.read_csv(means_file, sep="\t")

    mean_col = f"Mean fac{fac_num}"

    if "group" not in means_df.columns:
        raise ValueError(f"'group' column missing from means file: {means_file}")

    if mean_col not in means_df.columns:
        raise ValueError(f"'{mean_col}' column missing from means file: {means_file}")

    means_df["group"] = means_df["group"].astype(str).str.strip()

    means_dfs[factor_col] = means_df

means_dfs

{'fac1':   Effect           group     N  Mean fac1    SD fac1
 0  group           human  1000     24.693  14.953540
 1  group  persona_gemini  1000      3.998   2.932677
 2  group     persona_gpt  1000     12.108   7.512175
 3  group    persona_grok  1000      4.569   3.391274,
 'fac2':   Effect           group     N  Mean fac2    SD fac2
 0  group           human  1000     20.725  11.471660
 1  group  persona_gemini  1000      7.991   4.293874
 2  group     persona_gpt  1000     25.534  12.232354
 3  group    persona_grok  1000      9.008   5.235776,
 'fac3':   Effect           group     N  Mean fac3   SD fac3
 0  group           human  1000     14.831  9.920830
 1  group  persona_gemini  1000      5.007  3.917435
 2  group     persona_gpt  1000     11.757  7.446435
 3  group    persona_grok  1000      5.039  3.955898,
 'fac4':   Effect           group     N  Mean fac4   SD fac4
 0  group           human  1000      8.185  6.647327
 1  group  persona_gemini  1000      2.914  3.321391
 

## Simulate selection process

### List factors

In [7]:
factor_cols = [col for col in scores_file_ids_df.columns if re.fullmatch(r"fac\d+", col)]
factor_cols = sorted(factor_cols, key=lambda col: int(col.replace("fac", "")))

print("\n".join(
    f"{'' if i == 0 else '#'}FACTOR = {factor!r}"
    for i, factor in enumerate(factor_cols)
))

FACTOR = 'fac1'
#FACTOR = 'fac2'
#FACTOR = 'fac3'
#FACTOR = 'fac4'


### Copy/Paste the factors list and Uncomment/Comment factors and poles accordingly

In [35]:
#FACTOR = 'fac1'
#FACTOR = 'fac2'
FACTOR = 'fac3'
#FACTOR = 'fac4'

POLE = "positive"
#POLE = "negative"

sorted_means_dfs = means_dfs[FACTOR].sort_values(f"Mean {FACTOR}", ascending=POLE == "negative")

display(sorted_means_dfs)

groups_in_order = sorted_means_dfs["group"].tolist()

print("\n".join(
    f"{'' if i == 0 else '#'}GROUP = {group!r}"
    for i, group in enumerate(groups_in_order)
))

,Effect,group,N,Mean fac3,SD fac3
0,group,human,1000,14.831,9.920830
2,group,persona_gpt,1000,11.757,7.446435
3,group,persona_grok,1000,5.039,3.955898
1,group,persona_gemini,1000,5.007,3.917435


GROUP = 'human'
#GROUP = 'persona_gpt'
#GROUP = 'persona_grok'
#GROUP = 'persona_gemini'


### Copy/Paste the groups list and Uncomment/Comment groups accordingly

In `examples.py`, texts with a factor score of exactly `0` are skipped. In this simulation tool, they are not, for the sake of simplicity and to allow for a broader view.

In [37]:
GROUP = 'human'
#GROUP = 'persona_gpt'
#GROUP = 'persona_grok'
#GROUP = 'persona_gemini'

N_EXAMPLES = 40 # Define the number of examples to select

group_df = scores_file_ids_df.loc[scores_file_ids_df["group"].eq(GROUP)]

if POLE == "positive":
    examples_df = group_df.nlargest(N_EXAMPLES, FACTOR)
elif POLE == "negative":
    examples_df = group_df.nsmallest(N_EXAMPLES, FACTOR)
else:
    raise ValueError(f"Unknown POLE: {POLE!r}")

examples_df = examples_df[["group_filename", "group"] + factor_cols]

examples_df

,group_filename,group,fac1,fac2,fac3,fac4
3572,t572_human.txt,human,41,21,70,7
3157,t157_human.txt,human,23,22,62,10
3183,t183_human.txt,human,23,47,50,3
3038,t039_human.txt,human,26,62,49,18
3168,t168_human.txt,human,55,42,48,17
3386,t386_human.txt,human,38,45,48,23
3393,t393_human.txt,human,19,30,48,3
3513,t513_human.txt,human,39,32,48,12
3158,t158_human.txt,human,26,26,47,3
3605,t605_human.txt,human,70,32,47,8


## Fetch information from `examples/`

Write a Jupyter Notebook cell that parses generated LaTeX example files under `examples/` and builds a pandas DataFrame.

Requirements:

- Search only immediate subdirectories of `examples/` whose names match `f<n>_<pos|neg>`.
- Sort subdirectories by numeric factor number, then by pole order: `pos`, `neg`.
- In each matching subdirectory, parse only `.tex` files whose names match `f<n>_<pos|neg>_<m>.tex`, where the factor and pole match the parent directory.
- Sort files by numeric sequence `<m>`.
- For each file, read the first line that starts with `\begin{textsample}{`.
- Extract metadata from titles formatted as:

      POS Dim 1 – human – Score 101.00 – t451\_human.txt

  or:

      NEG Dim 2 – persona_gpt – Score -3.45 – t123\_gpt.txt

- Convert:
  - `POS` to `pole = "positive"`
  - `NEG` to `pole = "negative"`
  - `Dim <n>` to `factor = "fac<n>"`
  - escaped LaTeX underscores, such as `\_`, to `_` in `group` and `group_filename`
  - `Score <n>` to numeric `score`

- Create a pandas DataFrame with exactly these columns:

      factor
      pole
      group
      group_filename
      score

- The DataFrame should be sorted in the order of subdirectory/file parsing.
- Raise clear errors for matching files that lack a `\begin{textsample}{...}` line or whose metadata cannot be parsed.

In [10]:
import re
from pathlib import Path

import pandas as pd

EXAMPLES_DIR = Path("examples")

dir_pattern = re.compile(r"^f(?P<factor_num>\d+)_(?P<pole_code>pos|neg)$")
file_pattern_template = r"^f{factor_num}_{pole_code}_(?P<seq>\d+)\.tex$"

textsample_prefix = r"\begin{textsample}{"
metadata_pattern = re.compile(
    r"""^\\begin\{textsample\}\{
        (?P<pole_label>POS|NEG)
        \s+Dim\s+
        (?P<dim>\d+)
        \s+–\s+
        (?P<group>.+?)
        \s+–\s+
        Score\s+
        (?P<score>[+-]?(?:\d+(?:\.\d*)?|\.\d+))
        \s+–\s+
        (?P<group_filename>.+?)
        \}
    """,
    re.VERBOSE,
)

pole_name = {
    "POS": "positive",
    "NEG": "negative",
}

pole_order = {
    "pos": 0,
    "neg": 1,
}

rows = []

if not EXAMPLES_DIR.exists():
    raise FileNotFoundError(f"Examples directory not found: {EXAMPLES_DIR}")

example_dirs = []

for path in EXAMPLES_DIR.iterdir():
    if not path.is_dir():
        continue

    dir_match = dir_pattern.fullmatch(path.name)
    if not dir_match:
        continue

    example_dirs.append(
        {
            "path": path,
            "factor_num": int(dir_match.group("factor_num")),
            "pole_code": dir_match.group("pole_code"),
        }
    )

example_dirs = sorted(
    example_dirs,
    key=lambda item: (item["factor_num"], pole_order[item["pole_code"]]),
)

for example_dir in example_dirs:
    factor_num = example_dir["factor_num"]
    pole_code = example_dir["pole_code"]
    dir_path = example_dir["path"]

    file_pattern = re.compile(
        file_pattern_template.format(
            factor_num=factor_num,
            pole_code=pole_code,
        )
    )

    tex_files = []

    for tex_path in dir_path.iterdir():
        if not tex_path.is_file():
            continue

        file_match = file_pattern.fullmatch(tex_path.name)
        if not file_match:
            continue

        tex_files.append(
            {
                "path": tex_path,
                "seq": int(file_match.group("seq")),
            }
        )

    tex_files = sorted(tex_files, key=lambda item: item["seq"])

    for tex_file in tex_files:
        tex_path = tex_file["path"]

        textsample_line = None

        with open(tex_path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line.startswith(textsample_prefix):
                    textsample_line = line
                    break

        if textsample_line is None:
            raise ValueError(
                f"No line beginning with {textsample_prefix!r} found in {tex_path}"
            )

        metadata_match = metadata_pattern.match(textsample_line)

        if not metadata_match:
            raise ValueError(
                "Could not parse textsample metadata in "
                f"{tex_path}:\n{textsample_line}"
            )

        pole_label = metadata_match.group("pole_label")
        dim = int(metadata_match.group("dim"))

        if dim != factor_num:
            raise ValueError(
                f"Factor mismatch in {tex_path}: "
                f"directory says f{factor_num}, metadata says Dim {dim}"
            )

        expected_pole_label = pole_code.upper()
        if pole_label != expected_pole_label:
            raise ValueError(
                f"Pole mismatch in {tex_path}: "
                f"directory says {pole_code}, metadata says {pole_label}"
            )

        group = metadata_match.group("group").replace(r"\_", "_").strip()
        group_filename = metadata_match.group("group_filename").replace(r"\_", "_").strip()
        score = float(metadata_match.group("score"))

        rows.append(
            {
                "factor": f"fac{dim}",
                "pole": pole_name[pole_label],
                "group": group,
                "group_filename": group_filename,
                "score": score,
            }
        )

examples_summary_df = pd.DataFrame(
    rows,
    columns=[
        "factor",
        "pole",
        "group",
        "group_filename",
        "score",
    ],
)

# Export to NDJSON
from pathlib import Path

NDJSON_OUT = Path("examples/examples_summary.ndjson")

examples_summary_df.to_json(
    NDJSON_OUT,
    orient="records",
    lines=True,
    force_ascii=False,
)

print(f"Wrote {len(examples_summary_df):,} records to {NDJSON_OUT}")

examples_summary_df

Wrote 400 records to examples/examples_summary.ndjson


,factor,pole,group,group_filename,score
0,fac1,positive,human,t451_human.txt,101.0
1,fac1,positive,human,t998_human.txt,99.0
2,fac1,positive,human,t280_human.txt,99.0
3,fac1,positive,human,t023_human.txt,92.0
4,fac1,positive,human,t226_human.txt,91.0
...,...,...,...,...,...
395,fac4,negative,human,t831_human.txt,1.0
396,fac4,negative,human,t809_human.txt,1.0
397,fac4,negative,human,t796_human.txt,1.0
398,fac4,negative,human,t797_human.txt,1.0
